# Import Packages

In [1]:
import math
from collections import Counter
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer

# Load dataset and Tokenizer

Loading the dataset, and setup a tokenizer, using <i>bert-base-multilingual-cased</i>. 

In [2]:
dataset = load_dataset("coastalcph/tydi_xor_rc")
df_train = dataset["train"].to_pandas()
df_validation = dataset["validation"].to_pandas()
df_train_ko = df_train[df_train['lang'] == 'ko']
df_val_ko = df_validation[df_validation['lang'] == 'ko']

tokenizer = AutoTokenizer.from_pretrained(
    "bert-base-multilingual-cased", use_fast=True
)


# Tokenize questions and create vocabulary

Tokenizing questions and creating a vocabulary and only keeping tokens which appears four times or more in the dictionary, additionally adding the \<s\>, \<\/s\>, \<unk\> tokens to the vocabulary. 

Questions containing tokens which does not appear in the pruned vocabulary, said tokens will be switched out with the \<unk\> token. 


In [3]:
def tokenizeQuestion(df):
    return [tokenizer.tokenize(question) for question in df['question']]

t_questions = tokenizeQuestion(df_train_ko)
vocabulary = Counter(token for question in t_questions for token in question)
vocab = {token for token, count in vocabulary.items() if count >= 4}
vocab.update(['<s>', '</s>', '<unk>'])

def applyUnk(tokenizedQuestions, vocab):
    return [[token if token in vocab else '<unk>' for token in q]
            for q in tokenizedQuestions]

t_questions_filtered = applyUnk(t_questions, vocab)
val_unk = applyUnk(tokenizeQuestion(df_val_ko), vocab)

print('Training questions:', len(t_questions_filtered))
print('Validation questions:', len(val_unk))
print('Vocabulary size (including special tokens):', len(vocab))

Training questions: 2422
Validation questions: 356
Vocabulary size (including special tokens): 920


# Unigram Baseline Model implementation

In [4]:
class Unigram():
	def __init__ (self):
		self.count = Counter(token for question in t_questions_filtered for token in question + ['</s>'])
		self.total = sum(self.count.values())

	def probability(self, token):
		return self.count[token] / self.total

	def perplexity(self, questions):
		log_probability = 0
		token_count = 0
		for question in questions:
			for token in question + ["</s>"]:
				log_probability += math.log(self.probability(token))
				token_count += 1
		return math.exp(-log_probability / token_count)

unigram = Unigram()


unigram_ppl = unigram.perplexity(val_unk)
print(f'Unigram validation perplexity: {unigram_ppl:.2f}')


Unigram validation perplexity: 185.07


# Smoothed Bigram

Implementing a smoothed bigram model, using <i>add-1</i> smoothing

In [ ]:
class Bigram():
	def __init__(self):
		self.bigram_count = Counter()
		self.context_count = Counter()

	def train(self, sentences):
		for sentence in sentences:
			tokens = ['<s>'] + sentence + ['</s>']
			for i in range(1, len(tokens)):
				context = tokens[i-1]
				token = tokens[i]
				self.bigram_count[(context, token)] += 1
				self.context_count[context] += 1

	def probability(self, token, previous):
		return (self.bigram_count[(previous, token)] + 1) / (self.context_count[previous] + len(vocab))

	def perplexity(self, questions):
		log_probability = 0
		token_count = 0
		for question in questions:
			tokens = ['<s>'] + question + ['</s>']
			for previous, token in zip(tokens, tokens[1:]):
				log_probability += math.log(self.probability(token, previous))
				token_count += 1
		return math.exp(-log_probability / token_count)

bigram = Bigram()
bigram.train(t_questions_filtered)
bigram_ppl = bigram.perplexity(val_unk)

print(f'Smoothed bigram validation perplexity: {bigram_ppl:.2f}')


Smoothed bigram validation perplexity: 52.21


In [17]:
class Trigram():
	def __init__(self):
		self.trigram_count = Counter()
		self.bigram_count = Counter()



	def train(self, sentences):
		for sentence in sentences:
			tokens = ["<s>", "<s>"] + sentence + ["</s>"]
			for i in range(2, len(tokens)):
				context = (tokens[i-2], tokens[i-1])
				token = tokens[i]
				self.trigram_count[(context, token)] += 1
				self.bigram_count[context] += 1
    
	def probability(self, token, t1, t2, alpha):
		context = (t1, t2)
		numerator = self.trigram_count[(context, token)] + alpha
		denominator = self.bigram_count[context] + (alpha * len(vocab))
		if denominator == 0:
			return 0
		return numerator / denominator
    
	def perplexity(self, sentences, alpha):
		log_prob = 0
		total_tokens = 0
		for sentence in sentences:
			tokens = ["<s>","<s>"] + sentence + ["</s>"]
			for i in range(2, len(tokens)):
				t1 = tokens[i-2]
				t2 = tokens[i-1]
				token = tokens[i]
				p = self.probability(token, t1, t2, alpha)
				if p == 0:
					return float("inf")
				log_prob += math.log(p)
				total_tokens += 1
		
		if total_tokens == 0:
			return float("inf")
		return math.exp(-log_prob / total_tokens)



trigram = Trigram()
trigram.train(t_questions_filtered)

for alpha in [1, 0.1, 0.01, 0.001]:
    ppl = trigram.perplexity(val_unk, alpha=alpha)
    print(f"Trigram validation perplexity at alpha {alpha}: {ppl:.2f}")


Trigram validation perplexity at alpha 1: 103.84
Trigram validation perplexity at alpha 0.1: 43.01
Trigram validation perplexity at alpha 0.01: 25.76
Trigram validation perplexity at alpha 0.001: 24.73
